# 2018 and 2022 Elections

Download: [Secretary of State](https://sos.ga.gov/page/historical-elections-results)

The file must be downloaded manually, because the Secretary of State's website seems to have a robot filter.

In [ ]:
import zipfile
from pathlib import Path
import os

archivename = './data/november_6_2018_-_general_election.zip'
#archivename = './data/November 8, 2022 - General-Special Election.zip'
dirname = Path('./data/NovGen2018/November 6, 2018 - General Election/detail/xml/')
#dirname = Path('./data/NovGen2022/November 8, 2022 - General-Special Election/detail/xml/')
year = 2018

with zipfile.ZipFile(archivename) as archive:
    archive.extractall(f'./data/NovGen{year}')

metro_counties = ['Cherokee', 'Clayton', 'Cobb', 'DeKalb', 'Douglas', 'Fayette', 'Forsyth', 'Fulton', 'Gwinnett', 'Hall', 'Henry', 'Newton', 'Rockdale']

for filename in os.listdir(dirname):
    if any([county in filename for county in metro_counties]):
        county = filename.split('_')[0]
        with zipfile.ZipFile(dirname / filename) as file:
            file.extractall(f'./data/{county}{year}/')


In [250]:
import pandas as pd

import data
import utils

def retrieve(year):
    main_index = pd.MultiIndex.from_tuples([], names=['County', 'Precinct', 'Vote Method'])
    vote_files = {}

    for county in metro_counties:
        path = f'./data/{county}{year}/detail.xml'
        contests = pd.read_xml(path, xpath='//Contest')
        for contest in contests['text'].unique():
            if '/' in contest:
                contest_san = contest.split('/')[0]
            else:
                contest_san = contest
            if contest_san not in data.statewide_races_2022:
                continue
            if contest_san not in vote_files:
                vote_files[contest_san] = pd.DataFrame(index=main_index)
            choices = pd.read_xml(path, xpath=f'//Contest[@text="{contest}"]//Choice')
            for key,choice in zip(choices['key'], choices['text']):
                if choice == 'Shane Hazel (L)':
                    choice_san = 'Shane Hazel (Lib)'
                else:
                    choice_san = choice
                vote_types = pd.read_xml(path, xpath=f'//Contest[@text="{contest}"]//Choice[@key="{key}"]//VoteType')['name'].unique()
                for vote_type in vote_types:
                    try:
                        precincts = pd.read_xml(path, xpath=f'//Contest[@text="{contest}"]//Choice[@key="{key}"]//VoteType[@name="{vote_type}"]//Precinct')
                        index = pd.MultiIndex.from_tuples(utils.make_tuple_matrix(precincts['name'], [vote_type]), names=['Precinct', 'Vote Method'])
                        index = index.to_frame()
                        index.insert(0, 'County', [county for _ in index.index])
                        index = pd.MultiIndex.from_frame(index)
                        precincts = pd.DataFrame(precincts['votes'].tolist(), index=index, columns=['votes'])
                        target = vote_files[contest_san]
                        target = target.reindex(target.index.union(index))
                        target.loc[index, choice_san] = precincts['votes']
                        vote_files[contest_san] = target
                    except Exception as e:
                        print(contest)
                        print(choice)
                        print(vote_type)
                        raise e

    return vote_files

In [251]:
vote_files_2022 = retrieve(2022)
vote_files_2022['Governor']

Brian Kemp (I) (Rep)  \
County   Precinct    Vote Method                                    
Cherokee Air Acres   Absentee by Mail Votes                  76.0   
                     Advance Voting Votes                   538.0   
                     Election Day Votes                     658.0   
                     Provisional Votes                        0.0   
         Arnold Mill Absentee by Mail Votes                 113.0   
...                                                           ...   
Rockdale SP          Provisional Votes                        0.0   
         ST          Absentee by Mail Votes                  58.0   
                     Advance Voting Votes                   452.0   
                     Election Day Votes                     157.0   
                     Provisional Votes                        1.0   

                                             Stacey Abrams (Dem)  \
County   Precinct    Vote Method                                   
Cherokee Air Acres   Absentee by Mail Votes                 69.0   
                     Advance Voting Votes                  311.0   
                     Election Day Votes                    224.0   
                     Provisional Votes                       0.0   
         Arnold Mill Absentee by Mail Votes                103.0   
...                                                          ...   
Rockdale SP          Provisional Votes                       1.0   
         ST          Absentee by Mail Votes                 68.0   
                     Advance Voting Votes                  894.0   
                     Election Day Votes                    206.0   
                     Provisional Votes                       1.0   

                                             Shane Hazel (Lib)  
County   Precinct    Vote Method                                
Cherokee Air Acres   Absentee by Mail Votes                0.0  
                     Advance Voting Votes                  8.0  
                     Election Day Votes                   18.0  
                     Provisional Votes                     0.0  
         Arnold Mill Absentee by Mail Votes                2.0  
...                                                        ...  
Rockdale SP          Provisional Votes                     0.0  
         ST          Absentee by Mail Votes                1.0  
                     Advance Voting Votes                  3.0  
                     Election Day Votes                    9.0  
                     Provisional Votes                     0.0  

[4972 rows x 3 columns]

In [252]:
vote_files_2018 = retrieve(2018)
vote_files_2018['Governor']

BRIAN KEMP  (REP)  \
County   Precinct    Vote Method                            
Cherokee AIR ACRES   Absentee by Mail                82.0   
                     Advance in Person              496.0   
                     Election Day                   637.0   
                     Provisional                      1.0   
         ARNOLD MILL Absentee by Mail                91.0   
...                                                   ...   
Rockdale Stanton     Provisional                      0.0   
         The Lakes   Absentee by Mail                 6.0   
                     Advance in Person              171.0   
                     Election Day                   110.0   
                     Provisional                      0.0   

                                        STACEY ABRAMS  (DEM)  TED METZ (LIB)  
County   Precinct    Vote Method                                              
Cherokee AIR ACRES   Absentee by Mail                   41.0             0.0  
                     Advance in Person                 237.0             7.0  
                     Election Day                      321.0            27.0  
                     Provisional                         1.0             0.0  
         ARNOLD MILL Absentee by Mail                   64.0             4.0  
...                                                      ...             ...  
Rockdale Stanton     Provisional                         1.0             0.0  
         The Lakes   Absentee by Mail                   55.0             0.0  
                     Advance in Person                 377.0             0.0  
                     Election Day                      258.0             8.0  
                     Provisional                         1.0             0.0  

[6018 rows x 3 columns]

In [253]:
# Sanitization (2018)

def sanitize_2018(vote_data: pd.DataFrame):
    mapping = {'Advance in Person': 'Advance Voting', 
               'Advance in Person 1': 'Advance Voting', 
               'Advance in Person 2': 'Advance Voting', 
               'Advance in Person 3': 'Advance Voting'}
    vote_data = vote_data.rename(mapping, level='Vote Method')
    vote_data = vote_data.groupby(level=vote_data.index.names).sum()
    return vote_data

vote_files_2018 = {race: sanitize_2018(vote_data) for race, vote_data in vote_files_2018.items()}
vote_files_2018['Governor']

BRIAN KEMP  (REP)  \
County   Precinct    Vote Method                           
Cherokee AIR ACRES   Absentee by Mail               82.0   
                     Advance Voting                496.0   
                     Election Day                  637.0   
                     Provisional                     1.0   
         ARNOLD MILL Absentee by Mail               91.0   
...                                                  ...   
Rockdale Stanton     Provisional                     0.0   
         The Lakes   Absentee by Mail                6.0   
                     Advance Voting                171.0   
                     Election Day                  110.0   
                     Provisional                     0.0   

                                       STACEY ABRAMS  (DEM)  TED METZ (LIB)  
County   Precinct    Vote Method                                             
Cherokee AIR ACRES   Absentee by Mail                  41.0             0.0  
                     Advance Voting                   237.0             7.0  
                     Election Day                     321.0            27.0  
                     Provisional                        1.0             0.0  
         ARNOLD MILL Absentee by Mail                  64.0             4.0  
...                                                     ...             ...  
Rockdale Stanton     Provisional                        1.0             0.0  
         The Lakes   Absentee by Mail                  55.0             0.0  
                     Advance Voting                   377.0             0.0  
                     Election Day                     258.0             8.0  
                     Provisional                        1.0             0.0  

[4588 rows x 3 columns]

In [254]:
# Write to file (2018)

for race, vote_data in vote_files_2018.items():
    out_file_name = f'./data/2018_{race.lower().replace(" ", "_")}_metro.csv'
    vote_data.to_csv(out_file_name)

In [255]:
# Sanitization (2022)

def sanitize_2022(vote_data: pd.DataFrame):
    vote_data = vote_data.rename({f: f.replace(' Votes', '') for f in vote_data.index.get_level_values('Vote Method')}, level='Vote Method')
    return vote_data

vote_files_2022 = {race: sanitize_2022(vote_data) for race, vote_data in vote_files_2022.items()}
vote_files_2022['Governor']

    

Brian Kemp (I) (Rep)  \
County   Precinct    Vote Method                              
Cherokee Air Acres   Absentee by Mail                  76.0   
                     Advance Voting                   538.0   
                     Election Day                     658.0   
                     Provisional                        0.0   
         Arnold Mill Absentee by Mail                 113.0   
...                                                     ...   
Rockdale SP          Provisional                        0.0   
         ST          Absentee by Mail                  58.0   
                     Advance Voting                   452.0   
                     Election Day                     157.0   
                     Provisional                        1.0   

                                       Stacey Abrams (Dem)  Shane Hazel (Lib)  
County   Precinct    Vote Method                                               
Cherokee Air Acres   Absentee by Mail                 69.0                0.0  
                     Advance Voting                  311.0                8.0  
                     Election Day                    224.0               18.0  
                     Provisional                       0.0                0.0  
         Arnold Mill Absentee by Mail                103.0                2.0  
...                                                    ...                ...  
Rockdale SP          Provisional                       1.0                0.0  
         ST          Absentee by Mail                 68.0                1.0  
                     Advance Voting                  894.0                3.0  
                     Election Day                    206.0                9.0  
                     Provisional                       1.0                0.0  

[4972 rows x 3 columns]

In [256]:
# Write to file (2022)

for race, vote_data in vote_files_2022.items():
    out_file_name = f'./data/2022_{race.lower().replace(" ", "_")}_metro.csv'
    vote_data.to_csv(out_file_name)